# Semi-closed policy analysis

Edit the named run directories below and run all cells. The first run is the default baseline. The shared analysis functions read outputs without changing them.

Request plots cover completed, non-warmup rows. Family makespan includes recorded non-LLM delays and child dependencies. Incomplete families remain visible in coverage tables. Each run stays separate; these are descriptive statistics, not confidence intervals.

In [ ]:
from pathlib import Path

REPOSITORY = Path(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'research/analysis/semiclosed.py').is_file()))
RUNS = {
    'baseline': REPOSITORY / 'research/outputs/semiclosed/config-glm52-smoke-20261104',
}
BASELINE = 'baseline'

In [ ]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
if str(REPOSITORY) not in sys.path:
    sys.path.insert(0, str(REPOSITORY))
from research.analysis.semiclosed import load_runs, run_summary, ttft_statistics, paired_families
from research.analysis.plots import plot_ttft, plot_request_latency, plot_lifetimes, plot_family_timeline, plot_paired, plot_server_series

runs = load_runs(RUNS)
summary = pd.json_normalize([run_summary(run) for run in runs])
display(summary[['run', 'requests', 'completed_nonwarmup_rows', 'families', 'complete_families', 'source_complete', 'accounting_match', 'decomposition_available', 'decomposition_reason', 'notices']])

## Explore normalized data

These tables retain request identity, run labels, units, and eligibility. Use `requests`, `families`, `sessions`, and `summary` for additional plots. Raw engine E2E includes completion overhead; it is not the measured aggregate E2E definition.

In [ ]:
requests = pd.concat([run.requests for run in runs], ignore_index=True)
families = pd.concat([run.families for run in runs], ignore_index=True)
sessions = pd.concat([run.sessions for run in runs], ignore_index=True)
display(requests.head())
display(families.head())

## TTFT attribution

Admission plus scheduling delay includes host preprocessing and waiting. Remaining TTFT includes prefill and other first-token work. Neither is an isolated compute phase. Means add directly. Quantile bars interpolate components from the same total-TTFT-ranked requests, so the segments sum to total p50/p95/p99. Segments are not component percentiles. A missing/nonzero preemption count or invalid timing split produces an unsplit bar. KV fetch time is not currently recorded per request.

In [ ]:
ttft = pd.concat([ttft_statistics(run) for run in runs], ignore_index=True)
display(ttft)

In [ ]:
figure = plot_ttft(runs)
if figure is not None:
    display(figure)
    plt.close(figure)

In [ ]:
figure = plot_request_latency(runs)
if figure is not None:
    display(figure)
    plt.close(figure)

## Agent lifetimes and policy changes

Family makespan spans root arrival to the last completion in the family. Session lifetime spans that session's first arrival to last completion. Sample counts matter, especially for tail statistics. Pairing requires identical source requests, tokens, timestamps, dependency graphs, and source/converter provenance. Negative makespan change means improvement. Read the coverage table before interpreting paired curves.

In [ ]:
figure = plot_lifetimes(runs)
if figure is not None:
    display(figure)
    plt.close(figure)

In [ ]:
pairs, pair_coverage = paired_families(runs, BASELINE)
display(pair_coverage)
display(pairs.head())

In [ ]:
figure = plot_paired(runs, BASELINE)
if figure is not None:
    display(figure)
    plt.close(figure)

In [ ]:
figure = plot_family_timeline(runs)
if figure is not None:
    display(figure)
    plt.close(figure)

## Optional server time series

The run must have recorded `server_series`. Set `run.series_interval_s` to a positive value in the semi-closed run config before simulation. Gauges below are time-weighted bin means. Missing time series cannot be recovered from aggregate metrics.

In [ ]:
for run in runs:
    figure = plot_server_series(run)
    if figure is None:
        print(f'{run.label}: server series unavailable')
    else:
        display(figure)
        plt.close(figure)